# AI Decision Intelligence — Dataset Preprocessing Pipeline

**Contributor:** Abdul Samad  
**Scope:** Dataset sourcing and preprocessing  
**Environment:** Google Colab  

This notebook preprocesses the **four datasets already selected for the project**, one block at a time:

1. **Online Retail** — the ZIP file in the team's GitHub repository.
2. **Sample Expenses** — CSV in the same GitHub folder.
3. **Sample Products / Inventory** — CSV in the same GitHub folder.
4. **Online Retail II** — official UCI Machine Learning Repository download.

## Outputs

Files are written to `/content/ai_decision_intelligence_preprocessing/processed/`:

- `online_retail_cleaned.csv`
- `online_retail_daily_sales.csv`
- `expenses_cleaned.csv`
- `products_cleaned.csv`
- `online_retail_II_cleaned.csv`
- `online_retail_II_daily_sales.csv`
- `preprocessing_summary.csv`

### Important decisions

- Original files are downloaded into a separate `raw/` directory and are not overwritten.
- Text whitespace, column names, data types, exact duplicate rows, and required-field problems are handled explicitly.
- Retail transactions with negative quantities or cancellation invoice numbers are **retained and flagged**. Deleting them silently could discard legitimate returns/cancellations. The daily sales output includes both gross positive sales and net transaction value.
- Customer IDs and optional text fields are not required for daily sales aggregation, so records with missing customer IDs are retained.
- The expense and product files are small synthetic sample datasets. Their results are useful for validating the pipeline, but not enough by themselves to support broad business conclusions.

Run the setup cell first, then run **Blocks 1–4 in order**. The notebook uses direct public URLs; the GitHub repository must be accessible without signing in.

In [ ]:
# Setup: install Excel reader, import libraries, and create workspace
!pip -q install openpyxl requests

from pathlib import Path
from io import BytesIO
import re
import zipfile
import requests
import pandas as pd
import numpy as np
from IPython.display import display

BASE_DIR = Path("/content/ai_decision_intelligence_preprocessing")
RAW_DIR = BASE_DIR / "raw"
PROCESSED_DIR = BASE_DIR / "processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# GitHub folder shown in the project repository screenshot.
GITHUB_RAW_BASE = (
    "https://raw.githubusercontent.com/1iqrashaheen/AI_decision_Intelligance/"
    "Abdul_Samad_Task/Dataset_Sourcing_and_Data_Cleaning/Original_Datasets/"
)
ONLINE_RETAIL_ZIP_URL = GITHUB_RAW_BASE + "online+retail.zip"
EXPENSES_CSV_URL = GITHUB_RAW_BASE + "sample-expenses.csv"
PRODUCTS_CSV_URL = GITHUB_RAW_BASE + "sample-products.csv"

# Official UCI dataset download endpoint for Online Retail II.
ONLINE_RETAIL_II_ZIP_URL = "https://archive.ics.uci.edu/static/public/502/online+retail+ii.zip"

# Stores one summary per dataset; rerunning a block replaces its prior summary.
PREPROCESSING_SUMMARIES = {}

SESSION = requests.Session()
SESSION.headers.update({"User-Agent": "AI-Decision-Intelligence-Preprocessing/1.0"})


def download_file(url, destination, description):
    """Download one source file, save the untouched bytes, and report a clear error if unavailable."""
    destination = Path(destination)
    print(f"Downloading {description}...")
    response = SESSION.get(url, timeout=(30, 240))
    response.raise_for_status()
    content = response.content
    if not content:
        raise ValueError(f"The download for {description} returned an empty response.")
    destination.write_bytes(content)
    print(f"Saved raw file: {destination.name} ({len(content) / (1024 * 1024):.2f} MB)")
    return content


def normalize_column_name(name):
    """Normalize column labels to lowercase snake_case."""
    name = str(name).strip()
    name = re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", name)
    name = re.sub(r"[^A-Za-z0-9]+", "_", name)
    return name.strip("_").lower()


def standardize_retail_columns(raw_df):
    """Standardize UCI Online Retail / Online Retail II column names and validate schema."""
    df = raw_df.copy()
    df = df.dropna(how="all").copy()
    df.columns = [normalize_column_name(c) for c in df.columns]
    aliases = {
        "invoiceno": "invoice_no",
        "invoice_no": "invoice_no",
        "stockcode": "stock_code",
        "stock_code": "stock_code",
        "description": "description",
        "quantity": "quantity",
        "invoicedate": "invoice_date",
        "invoice_date": "invoice_date",
        "unitprice": "unit_price",
        "unit_price": "unit_price",
        "customerid": "customer_id",
        "customer_id": "customer_id",
        "country": "country",
    }
    df = df.rename(columns={c: aliases.get(c, c) for c in df.columns})
    required = ["invoice_no", "stock_code", "quantity", "invoice_date", "unit_price"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(
            f"Retail data is missing required columns: {missing}. "
            f"Columns found: {list(df.columns)}"
        )
    for optional in ["description", "customer_id", "country"]:
        if optional not in df.columns:
            df[optional] = pd.NA
    return df


def clean_retail_transactions(raw_df, dataset_key, output_filename):
    """Clean transaction fields while retaining returns and cancellations."""
    df = standardize_retail_columns(raw_df)
    rows_before = len(df)

    # Trim string fields. Treat IDs as strings so codes retain their identifiers.
    for col in ["invoice_no", "stock_code", "description", "country"]:
        df[col] = df[col].astype("string").str.strip()
        df[col] = df[col].replace({"": pd.NA})

    # Customer IDs can be missing and may be represented as floats in Excel.
    customer_numeric = pd.to_numeric(df["customer_id"], errors="coerce")
    def format_customer_id(value):
        if pd.isna(value):
            return pd.NA
        if float(value).is_integer():
            return str(int(value))
        return str(value).strip()
    df["customer_id"] = customer_numeric.map(format_customer_id).astype("string")

    df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")
    df["unit_price"] = pd.to_numeric(df["unit_price"], errors="coerce")
    df["invoice_date"] = pd.to_datetime(df["invoice_date"], errors="coerce", dayfirst=True)

    exact_duplicates = int(df.duplicated().sum())
    df = df.drop_duplicates().copy()

    # Rows missing transaction essentials cannot support valid transaction totals.
    required = ["invoice_no", "stock_code", "quantity", "invoice_date", "unit_price"]
    missing_required_rows = int(df[required].isna().any(axis=1).sum())
    df = df.dropna(subset=required).copy()
    df = df[(df["invoice_no"].str.len() > 0) & (df["stock_code"].str.len() > 0)].copy()

    # Keep unusual values and return/cancellation records, but make their meaning explicit.
    df["line_total_gbp"] = df["quantity"] * df["unit_price"]
    df["is_cancellation"] = df["invoice_no"].str.upper().str.startswith("C", na=False)
    df["is_negative_quantity"] = df["quantity"] < 0
    df["is_positive_sale"] = (
        (~df["is_cancellation"]) & (df["quantity"] > 0) & (df["unit_price"] > 0)
    )
    df["sales_date"] = df["invoice_date"].dt.normalize()
    df = df.sort_values(["invoice_date", "invoice_no", "stock_code"], kind="stable").reset_index(drop=True)

    output_path = PROCESSED_DIR / output_filename
    df.to_csv(output_path, index=False)

    # Daily series: gross positive sales plus net sales including negative returns/cancellations.
    net_daily = df.groupby("sales_date", dropna=False).agg(
        net_sales_gbp=("line_total_gbp", "sum"),
        net_quantity=("quantity", "sum"),
        transaction_lines=("invoice_no", "size"),
        invoices=("invoice_no", "nunique"),
    )
    positives = df[df["is_positive_sale"]]
    gross_daily = positives.groupby("sales_date").agg(
        gross_sales_gbp=("line_total_gbp", "sum"),
        units_sold=("quantity", "sum"),
        positive_sales_lines=("invoice_no", "size"),
        positive_sales_invoices=("invoice_no", "nunique"),
    )
    daily = net_daily.join(gross_daily, how="outer").sort_index()
    if not daily.empty:
        full_dates = pd.date_range(daily.index.min(), daily.index.max(), freq="D")
        daily = daily.reindex(full_dates)
        daily.index.name = "date"
        zero_cols = [
            "net_sales_gbp", "net_quantity", "transaction_lines", "invoices",
            "gross_sales_gbp", "units_sold", "positive_sales_lines", "positive_sales_invoices"
        ]
        daily[zero_cols] = daily[zero_cols].fillna(0)
        daily = daily.reset_index()
        daily["date"] = daily["date"].dt.strftime("%Y-%m-%d")
    else:
        daily = pd.DataFrame(columns=["date", "net_sales_gbp", "net_quantity", "transaction_lines", "invoices", "gross_sales_gbp", "units_sold", "positive_sales_lines", "positive_sales_invoices"])
    daily_output = output_filename.replace("_cleaned.csv", "_daily_sales.csv")
    daily.to_csv(PROCESSED_DIR / daily_output, index=False)

    PREPROCESSING_SUMMARIES[dataset_key] = {
        "dataset": dataset_key,
        "rows_loaded": rows_before,
        "exact_duplicate_rows_removed": exact_duplicates,
        "rows_missing_required_transaction_fields_removed": missing_required_rows,
        "rows_after_cleaning": len(df),
        "rows_with_missing_customer_id": int(df["customer_id"].isna().sum()),
        "cancellation_rows_retained": int(df["is_cancellation"].sum()),
        "negative_quantity_rows_retained": int(df["is_negative_quantity"].sum()),
        "cleaned_output": output_filename,
        "daily_sales_output": daily_output,
    }
    print(f"\n{dataset_key} preprocessing complete")
    print(f"Rows loaded: {rows_before:,} | rows after cleaning: {len(df):,}")
    print(f"Exact duplicate rows removed: {exact_duplicates:,}")
    print(f"Rows missing required transaction fields removed: {missing_required_rows:,}")
    print(f"Cancellation rows retained: {int(df['is_cancellation'].sum()):,}")
    print(f"Negative-quantity rows retained: {int(df['is_negative_quantity'].sum()):,}")
    print(f"Saved: {output_path}")
    print(f"Saved daily series: {PROCESSED_DIR / daily_output}")
    display(df.head())
    display(daily.head())
    return df, daily


def record_nonretail_summary(dataset_key, rows_loaded, duplicates_removed, required_rows_removed, cleaned_df, output_filename):
    PREPROCESSING_SUMMARIES[dataset_key] = {
        "dataset": dataset_key,
        "rows_loaded": int(rows_loaded),
        "exact_duplicate_rows_removed": int(duplicates_removed),
        "rows_missing_required_fields_removed": int(required_rows_removed),
        "rows_after_cleaning": int(len(cleaned_df)),
        "cleaned_output": output_filename,
    }

print("Workspace ready:", BASE_DIR)
print("Raw inputs will be kept separate from processed outputs.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

---
## Block 1 — Original Online Retail dataset (from the team's GitHub repository)

This uses `online+retail.zip` in the `Original_Datasets` folder shown in the repository. The code opens the archive and loads its Excel or CSV file without manually downloading/uploading it.

In [ ]:
# Download the original ZIP from the team's GitHub branch.
classic_zip_path = RAW_DIR / "online+retail.zip"
classic_bytes = download_file(ONLINE_RETAIL_ZIP_URL, classic_zip_path, "Online Retail ZIP from GitHub")

# Inspect archive contents and load the first supported data file.
if not zipfile.is_zipfile(BytesIO(classic_bytes)):
    raise ValueError(
        "The GitHub file did not download as a valid ZIP archive. Check that the repository is public "
        "and the file is still named 'online+retail.zip' in Original_Datasets."
    )

with zipfile.ZipFile(BytesIO(classic_bytes)) as zf:
    members = [n for n in zf.namelist() if not n.endswith("/") and not n.startswith("__MACOSX/")]
    supported = [n for n in members if n.lower().endswith((".xlsx", ".xls", ".csv"))]
    if not supported:
        raise ValueError(f"No Excel/CSV data file was found inside the ZIP. Archive members: {members[:20]}")
    chosen_member = supported[0]
    print("File found inside ZIP:", chosen_member)
    with zf.open(chosen_member) as inner_file:
        inner_bytes = inner_file.read()
    if chosen_member.lower().endswith(".csv"):
        raw_online_retail = pd.read_csv(BytesIO(inner_bytes), low_memory=False)
    else:
        raw_online_retail = pd.read_excel(BytesIO(inner_bytes), engine="openpyxl")

print("Raw shape:", raw_online_retail.shape)
display(raw_online_retail.head())

online_retail_cleaned, online_retail_daily = clean_retail_transactions(
    raw_online_retail,
    dataset_key="Online Retail",
    output_filename="online_retail_cleaned.csv",
)

print("\nDataset source: team repository")
print("Expected source page for the classic dataset: https://archive.ics.uci.edu/dataset/352/online+retail")

---
## Block 2 — Sample Expenses dataset (from the team's GitHub repository)

Cleaning includes normalized column names, trimmed text, parsed dates and amounts, removal of exact duplicate rows, and removal of records without the essential expense ID, date, or amount. Other records are retained where possible and reported.

In [ ]:
expenses_raw_path = RAW_DIR / "sample-expenses.csv"
expenses_bytes = download_file(EXPENSES_CSV_URL, expenses_raw_path, "Sample Expenses CSV from GitHub")
expenses_raw = pd.read_csv(BytesIO(expenses_bytes), low_memory=False)
expenses_raw.columns = [normalize_column_name(c) for c in expenses_raw.columns]
print("Raw shape:", expenses_raw.shape)
print("Columns:", list(expenses_raw.columns))
display(expenses_raw.head())

required_expense_columns = ["expense_id", "date", "amount"]
missing_expense_columns = [c for c in required_expense_columns if c not in expenses_raw.columns]
if missing_expense_columns:
    raise ValueError(f"Expenses CSV is missing columns {missing_expense_columns}; found {list(expenses_raw.columns)}")

expenses = expenses_raw.copy()
rows_before = len(expenses)
for col in expenses.select_dtypes(include=["object", "string"]).columns:
    expenses[col] = expenses[col].astype("string").str.strip().str.replace(r"\s+", " ", regex=True)
    expenses[col] = expenses[col].replace({"": pd.NA})

# Amount parsing tolerates currency symbols and thousands separators if present.
expenses["amount"] = pd.to_numeric(
    expenses["amount"].astype("string").str.replace(r"[^0-9.\-]", "", regex=True),
    errors="coerce",
)
expenses["date"] = pd.to_datetime(expenses["date"], errors="coerce")  # source uses YYYY-MM-DD
exact_duplicates = int(expenses.duplicated().sum())
expenses = expenses.drop_duplicates().copy()
required_missing = int(expenses[required_expense_columns].isna().any(axis=1).sum())
expenses = expenses.dropna(subset=required_expense_columns).copy()
expenses["year_month"] = expenses["date"].dt.to_period("M").astype("string")
expenses["is_negative_amount"] = expenses["amount"] < 0  # retained and flagged; can represent a reversal/adjustment
expenses["duplicate_expense_id"] = expenses["expense_id"].duplicated(keep=False)
expenses = expenses.sort_values(["date", "expense_id"], kind="stable").reset_index(drop=True)

expense_output = "expenses_cleaned.csv"
expenses.to_csv(PROCESSED_DIR / expense_output, index=False)
record_nonretail_summary("Sample Expenses", rows_before, exact_duplicates, required_missing, expenses, expense_output)

print("\nExpenses preprocessing complete")
print(f"Rows loaded: {rows_before:,} | rows after cleaning: {len(expenses):,}")
print(f"Exact duplicate rows removed: {exact_duplicates:,}")
print(f"Rows missing expense_id/date/amount removed: {required_missing:,}")
print("Duplicate expense IDs retained and flagged:", int(expenses["duplicate_expense_id"].sum()))
print("Saved:", PROCESSED_DIR / expense_output)
display(expenses.head())
display(expenses.isna().sum().rename("missing_values").to_frame())

---
## Block 3 — Sample Products / Inventory dataset (from the team's GitHub repository)

Cleaning standardizes labels and numeric fields, removes exact duplicate rows, and removes records missing a product ID, name, price, cost, or stock value. It adds useful inventory measures. The low-stock threshold is configurable; **10 units is only a default example**, not a project policy.

In [ ]:
products_raw_path = RAW_DIR / "sample-products.csv"
products_bytes = download_file(PRODUCTS_CSV_URL, products_raw_path, "Sample Products CSV from GitHub")
products_raw = pd.read_csv(BytesIO(products_bytes), low_memory=False)
products_raw.columns = [normalize_column_name(c) for c in products_raw.columns]
print("Raw shape:", products_raw.shape)
print("Columns:", list(products_raw.columns))
display(products_raw.head())

required_product_columns = ["product_id", "name", "price", "cost", "stock"]
missing_product_columns = [c for c in required_product_columns if c not in products_raw.columns]
if missing_product_columns:
    raise ValueError(f"Products CSV is missing columns {missing_product_columns}; found {list(products_raw.columns)}")

products = products_raw.copy()
rows_before = len(products)
for col in products.select_dtypes(include=["object", "string"]).columns:
    products[col] = products[col].astype("string").str.strip().str.replace(r"\s+", " ", regex=True)
    products[col] = products[col].replace({"": pd.NA})

for col in ["price", "cost", "stock", "rating"]:
    if col in products.columns:
        products[col] = pd.to_numeric(
            products[col].astype("string").str.replace(r"[^0-9.\-]", "", regex=True),
            errors="coerce",
        )

exact_duplicates = int(products.duplicated().sum())
products = products.drop_duplicates().copy()
required_missing = int(products[required_product_columns].isna().any(axis=1).sum())
products = products.dropna(subset=required_product_columns).copy()

# Derived fields support margin and inventory analysis; source values are not overwritten.
products["unit_margin"] = products["price"] - products["cost"]
products["gross_margin_pct"] = np.where(
    products["price"] > 0,
    (products["price"] - products["cost"]) / products["price"] * 100,
    np.nan,
)
products["inventory_cost_value"] = products["stock"] * products["cost"]
products["inventory_retail_value"] = products["stock"] * products["price"]
LOW_STOCK_THRESHOLD = 10  # adjust this if the team defines a different threshold
products["stock_status"] = np.select(
    [products["stock"] <= 0, products["stock"] <= LOW_STOCK_THRESHOLD],
    ["Out of stock", "Low stock"],
    default="In stock",
)
products["duplicate_product_id"] = products["product_id"].duplicated(keep=False)
if "sku" in products.columns:
    products["duplicate_sku"] = products["sku"].duplicated(keep=False)
if "rating" in products.columns:
    products["rating_out_of_range"] = ~products["rating"].between(0, 5) & products["rating"].notna()
products["negative_stock_flag"] = products["stock"] < 0
products = products.sort_values("product_id", kind="stable").reset_index(drop=True)

product_output = "products_cleaned.csv"
products.to_csv(PROCESSED_DIR / product_output, index=False)
record_nonretail_summary("Sample Products / Inventory", rows_before, exact_duplicates, required_missing, products, product_output)

print("\nProducts / Inventory preprocessing complete")
print(f"Rows loaded: {rows_before:,} | rows after cleaning: {len(products):,}")
print(f"Exact duplicate rows removed: {exact_duplicates:,}")
print(f"Rows missing product_id/name/price/cost/stock removed: {required_missing:,}")
print("Duplicate product IDs retained and flagged:", int(products["duplicate_product_id"].sum()))
print("Saved:", PROCESSED_DIR / product_output)
display(products.head())
display(products["stock_status"].value_counts(dropna=False).rename("product_count").to_frame())

---
## Block 4 — Online Retail II (official UCI download)

This block downloads Online Retail II from UCI, reads both workbook sheets, combines them into one transaction table, then applies the same retail cleaning rules as Block 1. The official source lists 1,067,371 instances and dates spanning December 2009 to December 2011. It is licensed under **CC BY 4.0**, so retain attribution if the data or derived files are shared.

In [ ]:


# ============================================================
# BLOCK 4: ONLINE RETAIL II - COMPLETE PREPROCESSING PIPELINE
# Source: UCI Machine Learning Repository
# Records: Approximately 1.07 million
# ============================================================

import os
import zipfile
import urllib.request
from pathlib import Path
from io import BytesIO

import pandas as pd
import numpy as np
from IPython.display import display

print("=" * 65)
print("ONLINE RETAIL II: DOWNLOAD AND PREPROCESSING")
print("=" * 65)

# ------------------------------------------------------------
# 1. CREATE OUTPUT FOLDERS
# ------------------------------------------------------------

raw_dir = Path("raw_datasets")
output_dir = Path("processed_datasets/online_retail_ii")

raw_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)

zip_path = raw_dir / "online_retail_II_source.zip"

dataset_url = (
    "https://archive.ics.uci.edu/static/public/502/"
    "online+retail+ii.zip"
)

# ------------------------------------------------------------
# 2. DOWNLOAD THE ORIGINAL DATASET
# ------------------------------------------------------------

if not zip_path.exists():
    print("\nDownloading Online Retail II...")
    urllib.request.urlretrieve(dataset_url, zip_path)
else:
    print("\nUsing existing downloaded ZIP file.")

print(f"ZIP size: {zip_path.stat().st_size / (1024**2):.2f} MB")

# ------------------------------------------------------------
# 3. EXTRACT AND READ THE EXCEL WORKBOOK
# ------------------------------------------------------------

with zipfile.ZipFile(zip_path, "r") as archive:
    excel_files = [
        name for name in archive.namelist()
        if name.lower().endswith((".xlsx", ".xls"))
    ]

    if not excel_files:
        raise FileNotFoundError("No Excel workbook found in the ZIP.")

    workbook_name = next(
        (name for name in excel_files
         if "online_retail_ii" in name.lower()),
        excel_files[0]
    )

    print(f"Workbook found: {workbook_name}")

    workbook_bytes = archive.read(workbook_name)

# Read both historical sheets
sheets = pd.read_excel(
    BytesIO(workbook_bytes),
    sheet_name=None
)

frames = []

for sheet_name, sheet_df in sheets.items():
    print(f"Sheet '{sheet_name}': {len(sheet_df):,} rows")
    frames.append(sheet_df)

raw_df = pd.concat(frames, ignore_index=True)

print("\nOriginal dataset shape:", raw_df.shape)
print("Original columns:", raw_df.columns.tolist())

# Save a raw CSV copy for reproducibility (optional).
# The original ZIP remains unchanged.
raw_csv_path = raw_dir / "online_retail_ii_raw.csv"
raw_df.to_csv(raw_csv_path, index=False)

# ------------------------------------------------------------
# 4. STANDARDIZE COLUMN NAMES
# FIX: Invoice -> invoice_no; Price -> unit_price
# ------------------------------------------------------------

df = raw_df.copy()

df.columns = [
    str(col).strip().lower().replace(" ", "_")
    for col in df.columns
]

column_mapping = {
    "invoice": "invoice_no",
    "invoiceno": "invoice_no",
    "stockcode": "stock_code",
    "invoicedate": "invoice_date",
    "price": "unit_price",
    "unitprice": "unit_price",
    "customerid": "customer_id",
}

df.rename(columns=column_mapping, inplace=True)

required_columns = [
    "invoice_no",
    "stock_code",
    "description",
    "quantity",
    "invoice_date",
    "unit_price",
    "country",
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}\n"
        f"Available columns: {df.columns.tolist()}"
    )

print("\nColumn standardization successful.")

# ------------------------------------------------------------
# 5. CLEAN TEXT FIELDS
# ------------------------------------------------------------

for col in ["invoice_no", "stock_code", "description", "country"]:
    df[col] = df[col].astype("string").str.strip()

# Convert empty strings to missing values
for col in ["description", "country", "stock_code", "invoice_no"]:
    df[col] = df[col].replace("", pd.NA)

# ------------------------------------------------------------
# 6. CONVERT DATA TYPES
# ------------------------------------------------------------

df["invoice_date"] = pd.to_datetime(
    df["invoice_date"],
    errors="coerce"
)

df["quantity"] = pd.to_numeric(
    df["quantity"],
    errors="coerce"
)

df["unit_price"] = pd.to_numeric(
    df["unit_price"],
    errors="coerce"
)

if "customer_id" in df.columns:
    df["customer_id"] = pd.to_numeric(
        df["customer_id"],
        errors="coerce"
    )

# ------------------------------------------------------------
# 7. ADD TRANSACTION FLAGS BEFORE FILTERING
# Preserve cancellation/return information.
# ------------------------------------------------------------

df["is_cancellation"] = (
    df["invoice_no"]
    .str.upper()
    .str.startswith("C", na=False)
)

df["is_return_or_negative_quantity"] = df["quantity"] < 0

df["is_negative_price"] = df["unit_price"] < 0

# Original signed transaction value
df["sales_value"] = df["quantity"] * df["unit_price"]

# Positive sales only: useful for gross sales forecasting
df["positive_sales_value"] = np.where(
    (df["quantity"] > 0)
    & (df["unit_price"] >= 0)
    & (~df["is_cancellation"]),
    df["sales_value"],
    0.0
)

# ------------------------------------------------------------
# 8. REPORT DATA QUALITY BEFORE FILTERING
# ------------------------------------------------------------

print("\nMissing values before filtering:")
print(df[required_columns].isna().sum())

rows_before = len(df)
duplicates_before = int(df.duplicated().sum())

print(f"\nRows before cleaning: {rows_before:,}")
print(f"Exact duplicate rows: {duplicates_before:,}")

# ------------------------------------------------------------
# 9. REMOVE EXACT DUPLICATES
# ------------------------------------------------------------

df = df.drop_duplicates().copy()

# ------------------------------------------------------------
# 10. REMOVE ROWS INVALID FOR TRANSACTION ANALYSIS
# Retain cancellations and returns if the transaction fields
# themselves are valid.
# ------------------------------------------------------------

valid_mask = (
    df["invoice_no"].notna()
    & df["stock_code"].notna()
    & df["invoice_date"].notna()
    & df["quantity"].notna()
    & df["unit_price"].notna()
    & df["country"].notna()
)

df = df.loc[valid_mask].copy()

# Description can be missing in the source, so do not discard
# otherwise valid transactions just because description is absent.

# Recalculate transaction value after data type conversion.
df["sales_value"] = df["quantity"] * df["unit_price"]

df["positive_sales_value"] = np.where(
    (df["quantity"] > 0)
    & (df["unit_price"] >= 0)
    & (~df["is_cancellation"]),
    df["sales_value"],
    0.0
)

# ------------------------------------------------------------
# 11. ADD TIME FEATURES FOR ANALYSIS
# ------------------------------------------------------------

df["sales_date"] = df["invoice_date"].dt.floor("D")
df["sales_year"] = df["invoice_date"].dt.year
df["sales_month"] = df["invoice_date"].dt.month
df["sales_day"] = df["invoice_date"].dt.day
df["sales_day_of_week"] = df["invoice_date"].dt.dayofweek

# ------------------------------------------------------------
# 12. GENERATE DAILY SALES TIME SERIES
# ------------------------------------------------------------

daily_sales = (
    df.groupby("sales_date", as_index=False)
    .agg(
        transaction_lines=("invoice_no", "size"),
        gross_positive_sales=("positive_sales_value", "sum"),
        net_sales=("sales_value", "sum"),
        units_sold=("quantity", lambda s: s[s > 0].sum()),
        return_units=("quantity", lambda s: -s[s < 0].sum()),
        cancellation_lines=("is_cancellation", "sum"),
    )
    .sort_values("sales_date")
)

# Include calendar days without transactions.
if not daily_sales.empty:
    full_dates = pd.date_range(
        start=daily_sales["sales_date"].min(),
        end=daily_sales["sales_date"].max(),
        freq="D"
    )

    daily_sales = (
        daily_sales.set_index("sales_date")
        .reindex(full_dates)
        .rename_axis("sales_date")
        .reset_index()
    )

    count_columns = [
        "transaction_lines",
        "units_sold",
        "return_units",
        "cancellation_lines",
    ]

    for col in count_columns:
        daily_sales[col] = daily_sales[col].fillna(0)

    for col in ["gross_positive_sales", "net_sales"]:
        daily_sales[col] = daily_sales[col].fillna(0.0)

# ------------------------------------------------------------
# 13. SAVE CLEANED OUTPUTS
# ------------------------------------------------------------

cleaned_path = output_dir / "online_retail_ii_cleaned.csv"
daily_path = output_dir / "online_retail_ii_daily_sales.csv"
quality_path = output_dir / "online_retail_ii_quality_report.csv"

df.to_csv(cleaned_path, index=False)
daily_sales.to_csv(daily_path, index=False)

quality_report = pd.DataFrame({
    "metric": [
        "original_rows",
        "exact_duplicates_removed",
        "rows_after_duplicate_removal",
        "rows_after_required_field_filter",
        "rows_removed_by_required_field_filter",
        "cancellation_rows_retained",
        "negative_quantity_rows_retained",
        "missing_customer_ids",
        "cleaned_rows",
        "daily_time_series_rows",
    ],
    "value": [
        rows_before,
        duplicates_before,
        rows_before - duplicates_before,
        len(df),
        rows_before - duplicates_before - len(df),
        int(df["is_cancellation"].sum()),
        int(df["is_return_or_negative_quantity"].sum()),
        int(df["customer_id"].isna().sum())
            if "customer_id" in df.columns else None,
        len(df),
        len(daily_sales),
    ]
})

quality_report.to_csv(quality_path, index=False)

# ------------------------------------------------------------
# 14. DISPLAY RESULTS
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PREPROCESSING COMPLETE")
print("=" * 65)

print(f"Original rows: {rows_before:,}")
print(f"Cleaned rows: {len(df):,}")
print(f"Daily sales records: {len(daily_sales):,}")

print("\nCleaned dataset preview:")
display(df.head())

print("\nDaily sales preview:")
display(daily_sales.head())

print("\nData quality report:")
display(quality_report)

print("\nSaved files:")
print(f"Raw CSV:      {raw_csv_path}")
print(f"Cleaned CSV:  {cleaned_path}")
print(f"Daily sales:  {daily_path}")
print(f"Quality CSV:  {quality_path}")

print("\nNote: Missing customer IDs are retained.")
print("Cancellations and negative-quantity transactions are flagged.")
print("Gross positive sales and net sales are provided separately.")

---
## Final check — Review outputs and create a summary

Run this cell after the blocks you need have completed. It summarizes row counts, missing values, and generated files. If a block was not run, its outputs will simply be absent.

In [ ]:
# Write summary and show every output created so far.
if PREPROCESSING_SUMMARIES:
    summary_df = pd.DataFrame(list(PREPROCESSING_SUMMARIES.values()))
    summary_path = PROCESSED_DIR / "preprocessing_summary.csv"
    summary_df.to_csv(summary_path, index=False)
    print("Preprocessing summary:")
    display(summary_df)
else:
    print("No preprocessing blocks have run yet.")

print("\nProcessed files:")
for output in sorted(PROCESSED_DIR.glob("*")):
    print(f"- {output.name} ({output.stat().st_size / (1024 * 1024):.2f} MB)")

print("\nRaw downloaded files remain in:", RAW_DIR)
print("Processed files are in:", PROCESSED_DIR)

In [ ]:
# Optional: package the cleaned outputs into one ZIP for downloading from Colab.
# Uncomment the last two lines to trigger a browser download when using Google Colab.
import zipfile
OUTPUT_ZIP = BASE_DIR / "processed_datasets_bundle.zip"
with zipfile.ZipFile(OUTPUT_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for output in sorted(PROCESSED_DIR.glob("*")):
        archive.write(output, arcname=output.name)
print("Prepared output bundle:", OUTPUT_ZIP)
# from google.colab import files
# files.download(str(OUTPUT_ZIP))

## Data-source and use notes

- **Classic Online Retail:** The file is taken from this project's `Original_Datasets` GitHub folder; dataset context and attribution: [UCI Online Retail](https://archive.ics.uci.edu/dataset/352/online+retail), Chen (2015), DOI [10.24432/C5BW33](https://doi.org/10.24432/C5BW33), CC BY 4.0.
- **Sample Expenses:** [HowToCSV Sample Expense CSV](https://howtocsv.com/sample-expenses-csv/). The local GitHub copy is loaded by the notebook; source describes it as synthetic sample data.
- **Sample Products:** [HowToCSV Sample Products CSV](https://howtocsv.com/sample-products-csv/). The local GitHub copy is loaded by the notebook; source describes it as synthetic sample data.
- **Online Retail II:** [UCI Online Retail II](https://archive.ics.uci.edu/dataset/502/online+retail), Chen (2012), DOI [10.24432/C5CG6D](https://doi.org/10.24432/C5CG6D), CC BY 4.0.

### Forecasting note

For forecasting, start with `online_retail_daily_sales.csv` or `online_retail_II_daily_sales.csv`. The `gross_sales_gbp` column counts positive non-cancellation lines; `net_sales_gbp` includes all retained transaction lines, including negative-value returns/cancellations. Choose which target to model with the team and document the decision. Use a chronological train/test split; do not randomly shuffle dates.